
# ExoMinerals — Phase 3: First Pass at the Modeling Approach (Supervised Proxy)

This part is the **first aspect of the modeling approach** from our Mapping plan:
- **Supervised learning proxy** - uses a small **Solar System reference set** to learn weights mapping astrophysical/planetary features to mineral group likelihoods.
- Applies those learned weights to **exoplanets** from the NASA Exoplanet Archive PS table (`PS.csv`).



## 1. Imports


In [12]:

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score

import matplotlib.pyplot as plt

# Display options
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)



## 2. Load Exoplanet PS table


In [13]:

file_path = "PS.csv"

df = pd.read_csv(
    file_path,
    comment="#",    # skip metadata lines starting with #
    sep=",",        # true CSV after header
    engine="python"
)

print(df.shape)
df.head()


(38898, 92)


,pl_name,hostname,default_flag,sy_snum,sy_pnum,discoverymethod,disc_year,disc_facility,soltype,pl_controv_flag,pl_refname,pl_orbper,pl_orbpererr1,pl_orbpererr2,pl_orbperlim,pl_orbsmax,pl_orbsmaxerr1,pl_orbsmaxerr2,pl_orbsmaxlim,pl_rade,pl_radeerr1,pl_radeerr2,pl_radelim,pl_radj,pl_radjerr1,pl_radjerr2,pl_radjlim,pl_bmasse,pl_bmasseerr1,pl_bmasseerr2,pl_bmasselim,pl_bmassj,pl_bmassjerr1,pl_bmassjerr2,pl_bmassjlim,pl_bmassprov,pl_orbeccen,pl_orbeccenerr1,pl_orbeccenerr2,pl_orbeccenlim,pl_insol,pl_insolerr1,pl_insolerr2,pl_insollim,pl_eqt,pl_eqterr1,pl_eqterr2,pl_eqtlim,ttv_flag,st_refname,st_spectype,st_teff,st_tefferr1,st_tefferr2,st_tefflim,st_rad,st_raderr1,st_raderr2,st_radlim,st_mass,st_masserr1,st_masserr2,st_masslim,st_met,st_meterr1,st_meterr2,st_metlim,st_metratio,st_logg,st_loggerr1,st_loggerr2,st_logglim,sy_refname,rastr,ra,decstr,dec,sy_dist,sy_disterr1,sy_disterr2,sy_vmag,sy_vmagerr1,sy_vmagerr2,sy_kmag,sy_kmagerr1,sy_kmagerr2,sy_gaiamag,sy_gaiamagerr1,sy_gaiamagerr2,rowupdate,pl_pubdate,releasedate
0,11 Com b,11 Com,1,2,1,Radial Velocity,2007,Xinglong Station,Published Confirmed,0,<a refstr=TENG_ET_AL__2023 href=https://ui.ads...,323.21,0.06,-0.05,0.0,1.178,0.00,0.00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4914.898486,39.092894,-39.728551,0.0,15.464,0.123,-0.125,0.0,Msini,0.238,0.007,-0.007,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,<a refstr=TENG_ET_AL__2023 href=https://ui.ads...,G8 III,4874.0,NaN,NaN,0.0,13.76,2.85,-2.45,0.0,2.09,0.64,-0.63,0.0,-0.26,0.10,-0.10,0.0,[Fe/H],2.45,0.08,-0.08,0.0,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,12h20m42.91s,185.178779,+17d47m35.71s,17.793252,93.1846,1.9238,-1.9238,4.72307,0.023,-0.023,2.282,0.346,-0.346,4.44038,0.003848,-0.003848,2023-09-19,2023-08,2023-09-19
1,11 Com b,11 Com,0,2,1,Radial Velocity,2007,Xinglong Station,Published Confirmed,0,<a refstr=KUNITOMO_ET_AL__2011 href=https://ui...,NaN,NaN,NaN,NaN,1.210,0.06,-0.05,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5434.700000,540.300000,-413.200000,0.0,17.100,1.700,-1.300,0.0,Msini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,<a refstr=KUNITOMO_ET_AL__2011 href=https://ui...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.60,0.40,-0.30,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,12h20m42.91s,185.178779,+17d47m35.71s,17.793252,93.1846,1.9238,-1.9238,4.72307,0.023,-0.023,2.282,0.346,-0.346,4.44038,0.003848,-0.003848,2014-07-23,2011-08,2014-07-23
2,11 Com b,11 Com,0,2,1,Radial Velocity,2007,Xinglong Station,Published Confirmed,0,<a refstr=LIU_ET_AL__2008 href=https://ui.adsa...,326.03,0.32,-0.32,0.0,1.290,0.05,-0.05,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6165.600000,476.700000,-476.700000,0.0,19.400,1.500,-1.500,0.0,Msini,0.231,0.005,-0.005,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,<a refstr=LIU_ET_AL__2008 href=https://ui.adsa...,G8 III,4742.0,100.0,-100.0,0.0,19.00,2.00,-2.00,0.0,2.70,0.30,-0.30,0.0,-0.35,0.09,-0.09,0.0,[Fe/H],2.31,0.10,-0.10,0.0,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,12h20m42.91s,185.178779,+17d47m35.71s,17.793252,93.1846,1.9238,-1.9238,4.72307,0.023,-0.023,2.282,0.346,-0.346,4.44038,0.003848,-0.003848,2014-05-14,2008-01,2014-05-14
3,11 UMi b,11 UMi,0,1,1,Radial Velocity,2009,Thueringer Landessternwarte Tautenburg,Published Confirmed,0,<a refstr=KUNITOMO_ET_AL__2011 href=https://ui...,NaN,NaN,NaN,NaN,1.510,0.06,-0.05,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3432.400000,381.400000,-413.200000,0.0,10.800,1.200,-1.300,0.0,Msini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,<a refstr=KUNITOMO_ET_AL__2011 href=https://ui...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.70,0.40,-0.30,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,15h17m05.90s,229.274595,+71d49m26.19s,71.823943,125.3210,1.9765,-1.9765,5.01300,0.005,-0.005,1.939,0.270,-0.270,4.56216,0.003903,-0.003903,2018-04-25,2011-08,2014-07-23
4,11 UMi b,11 UMi,0,1,1,Radial Velocity,2009,Thueringer Landessternwarte Tautenburg,Published Confirmed,0,<a refstr=DOLLINGER_ET_AL__2009 hre


## 3. Feature selection (top ~10, presence-checked)

We aim for 10 especially relevant predictors. The PS table field names vary across versions; this section
checks for common aliases.

1. `st_met` — stellar metallicity  
2. `st_teff` — stellar effective temperature (K)  
3. `st_mass` — stellar mass (solar)  
4. `pl_rade` — planet radius (Earth radii)  
5. `pl_bmasse` — planet mass (Earth masses)  
6. `pl_dens` or derived density (mass/volume) if possible  
7. `pl_orbper` — orbital period (days)  
8. `pl_orbsmax` — semi-major axis (AU)  
9. `pl_eqt` — planetary equilibrium temperature (K)  
10. `pl_insol` — incident stellar flux (Earth=1)


In [14]:

# Identify available columns or acceptable aliases
colmap = {
    'st_met': ['st_met', 'stellar_metallicity'],
    'st_teff': ['st_teff', 'stellar_teff', 'st_teff_k'],
    'st_mass': ['st_mass', 'stellar_mass', 'sy_msun'],
    'pl_rade': ['pl_rade', 'planet_radius_earth', 'pl_rade_e'],
    'pl_bmasse': ['pl_bmasse', 'pl_masse', 'planet_mass_earth'],
    'pl_dens': ['pl_dens', 'planet_density', 'pl_dens_earth'],
    'pl_orbper': ['pl_orbper', 'orbital_period', 'pl_orbper_days'],
    'pl_orbsmax': ['pl_orbsmax', 'semi_major_axis', 'pl_orbsmax_au'],
    'pl_eqt': ['pl_eqt', 'equilibrium_temp', 'pl_eqt_k'],
    'pl_insol': ['pl_insol', 'insolation', 'pl_insol_earth']
}

selected_cols = {}
for k, candidates in colmap.items():
    c = safe_col(df, candidates)
    if c is not None:
        selected_cols[k] = c

feature_cols = list(selected_cols.values())

print("Selected/derived feature columns found in PS.csv:")
feature_cols


Selected/derived feature columns found in PS.csv:


['st_met',
 'st_teff',
 'st_mass',
 'pl_rade',
 'pl_bmasse',
 'pl_orbper',
 'pl_orbsmax',
 'pl_eqt',
 'pl_insol']


## 4. Build a Solar System reference set (targets we **know**)

We create a compact Solar System table with:
- **Features** compatible with PS columns (same star for all: the Sun).
- **Targets**: mineral **group** proportions (coarse, for prototype):  
  - `iron_metal` (core/metal content proxy)  
  - `silicates` (mantle/crust rock-formers)  
  - `water_ice` (surface/subsurface ice fraction)  
  - `sulfates_carbonates` (evaporites/alteration minerals as a proxy for geochemical cycling)

> **Note**: Values below are *reasonable prototyping estimates* consolidated from well-known planetary science references.  
> Use them as placeholders; you should refine/replace with a vetted table when ready.


In [15]:

# Shared solar values (Sun)
SUN = {
    'st_met': 0.0,         # solar metallicity relative to Sun (by definition ~0 dex)
    'st_teff': 5772.0,     # K
    'st_mass': 1.0         # Msun
}

# Helper to estimate density in Earth units if mass & radius provided (both Earth units)
def dens_earth_units(m_e, r_e):
    if m_e is None or r_e is None or r_e == 0:
        return None
    return m_e / (r_e ** 3)

# Solar System bodies with reasonably known bulk characteristics
# radius (Earth radii), mass (Earth masses), orbital period (days), sma (AU), eqt (approx K), insol (Earth=1)
solsys_rows = [
    # name, rade, bmasse, orbper, sma, eqt, insol, targets: [iron_metal, silicates, water_ice, sulf/carbs]
    ("Mercury", 0.383, 0.055, 87.97, 0.387, 440, 6.67, [0.65, 0.35, 0.00, 0.00]),
    ("Venus",   0.949, 0.815, 224.70, 0.723, 737, 1.91, [0.32, 0.68, 0.00, 0.00]),
    ("Earth",   1.000, 1.000, 365.25, 1.000, 255, 1.00, [0.32, 0.68, 0.00, 0.05]),
    ("Moon",    0.273, 0.0123, 27.32, 0.00257, 220, 1.00, [0.03, 0.97, 0.00, 0.00]),
    ("Mars",    0.532, 0.107, 686.98, 1.524, 210, 0.43, [0.25, 0.70, 0.02, 0.03]),
    ("Ceres",   0.074, 0.00015, 1680.0, 2.77, 160, 0.13, [0.00, 0.40, 0.60, 0.00]),
    ("Io",      0.286, 0.015, 1.769, 0.00282, 110, 1.00, [0.30, 0.60, 0.00, 0.10]),
    ("Europa",  0.245, 0.008, 3.551, 0.00449, 102, 1.00, [0.10, 0.20, 0.70, 0.00]),
    ("Ganymede",0.413, 0.025, 7.155, 0.00716, 110, 1.00, [0.10, 0.30, 0.60, 0.00]),
    ("Callisto",0.378, 0.018, 16.689, 0.0126, 134, 1.00, [0.10, 0.20, 0.70, 0.00]),
    ("Titan",   0.404, 0.0225, 15.945, 0.00817, 94, 1.00, [0.05, 0.15, 0.75, 0.05]),
]

sol_df = pd.DataFrame(solsys_rows, columns=[
    "name","pl_rade","pl_bmasse","pl_orbper","pl_orbsmax","pl_eqt","pl_insol",
    "target_vec"
])

# Add stellar columns
sol_df['st_met'] = SUN['st_met']
sol_df['st_teff'] = SUN['st_teff']
sol_df['st_mass'] = SUN['st_mass']

# Derived density in Earth units
sol_df['pl_dens'] = [dens_earth_units(m, r) for r, m in zip(sol_df['pl_rade'], sol_df['pl_bmasse'])]

# Unpack vectors into columns
target_names = ['iron_metal','silicates','water_ice','sulfates_carbonates']
targets = np.vstack(sol_df['target_vec'].values)
for i, tname in enumerate(target_names):
    sol_df[tname] = targets[:, i]

# Reorder columns for clarity
sol_feature_cols = ['st_met','st_teff','st_mass','pl_rade','pl_bmasse','pl_dens','pl_orbper','pl_orbsmax','pl_eqt','pl_insol']
sol_df = sol_df[['name'] + sol_feature_cols + target_names]
sol_df.head(12)


,name,st_met,st_teff,st_mass,pl_rade,pl_bmasse,pl_dens,pl_orbper,pl_orbsmax,pl_eqt,pl_insol,iron_metal,silicates,water_ice,sulfates_carbonates
0,Mercury,0.0,5772.0,1.0,0.383,0.05500,0.978963,87.970,0.38700,440,6.67,0.65,0.35,0.00,0.00
1,Venus,0.0,5772.0,1.0,0.949,0.81500,0.953584,224.700,0.72300,737,1.91,0.32,0.68,0.00,0.00
2,Earth,0.0,5772.0,1.0,1.000,1.00000,1.000000,365.250,1.00000,255,1.00,0.32,0.68,0.00,0.05
3,Moon,0.0,5772.0,1.0,0.273,0.01230,0.604529,27.320,0.00257,220,1.00,0.03,0.97,0.00,0.00
4,Mars,0.0,5772.0,1.0,0.532,0.10700,0.710639,686.980,1.52400,210,0.43,0.25,0.70,0.02,0.03
5,Ceres,0.0,5772.0,1.0,0.074,0.00015,0.370166,1680.000,2.77000,160,0.13,0.00,0.40,0.60,0.00
6,Io,0.0,5772.0,1.0,0.286,0.01500,0.641199,1.769,0.00282,110,1.00,0.30,0.60,0.00,0.10
7,Europa,0.0,5772.0,1.0,0.245,0.00800,0.543991,3.551,0.00449,102,1.00,0.10,0.20,0.70,0.00
8,Ganymede,0.0,5772.0,1.0,0.413,0.02500,0.354887,7.155,0.00716,110,1.00,0.10,0.30,0.60,0.00
9,Callisto,0.0,5772.0,1.0,0.378,0.01800,0.333271,16.689,0.01260,134,1.00,0.10,0.20,0.70,0.00



## 5. Aligning features and building the supervised model

- We align the Solar System features to the feature set discovered in `PS.csv`.
- We train a **regularized linear model** (Ridge inside a MultiOutputRegressor) to learn weights from features → mineral groups.
- We use **Leave-One-Out CV** due to the small sample size to sanity-check generalization.


In [20]:

# Columns actually available in PS.csv (discovered earlier)
ps_feature_cols = feature_cols

# Solar System training X with only those columns (fallback: drop columns not present in PS)
train_feature_cols = []
for key, candidates in {
    'st_met': ['st_met'],
    'st_teff': ['st_teff'],
    'st_mass': ['st_mass'],
    'pl_rade': ['pl_rade'],
    'pl_bmasse': ['pl_bmasse'],
    'pl_dens': ['pl_dens'],
    'pl_orbper': ['pl_orbper'],
    'pl_orbsmax': ['pl_orbsmax'],
    'pl_eqt': ['pl_eqt'],
    'pl_insol': ['pl_insol']
}.items():
    for c in candidates:
        if c in ps_feature_cols and c in sol_df.columns:
            train_feature_cols.append(c)
            break

X_train = sol_df[train_feature_cols].copy()
Y_train = sol_df[['iron_metal','silicates','water_ice','sulfates_carbonates']].copy()

# Pipeline: impute -> scale -> Ridge (multi-output)
# The following code was developed with the aid of ChatGPT
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocess = ColumnTransformer(
    transformers=[('num', numeric_transformer, train_feature_cols)],
    remainder='drop'
)

base = Ridge(alpha=1.0, random_state=42)
model = Pipeline(steps=[('prep', preprocess),
                       ('reg', MultiOutputRegressor(base))])

# Leave-One-Out CV to get quick MAE scores
loo = LeaveOneOut()
mae_scores = []
pred_vs_true = []

for train_idx, test_idx in loo.split(X_train):
    model.fit(X_train.iloc[train_idx], Y_train.iloc[train_idx])
    pred = model.predict(X_train.iloc[test_idx])
    mae = np.mean(np.abs(pred - Y_train.iloc[test_idx].values))
    mae_scores.append(mae)
    pred_vs_true.append((sol_df.iloc[test_idx]['name'].values[0], pred[0], Y_train.iloc[test_idx].values[0]))

cv_mae = np.mean(mae_scores)
cv_mae


np.float64(0.37111568847266413)

In [23]:
# Fit on all Solar System data
model.fit(X_train, Y_train)


LOO-CV mean absolute error (average over mineral groups): 0.3711
Features used: ['st_met', 'st_teff', 'st_mass', 'pl_rade', 'pl_bmasse', 'pl_orbper', 'pl_orbsmax', 'pl_eqt', 'pl_insol']


,steps,"[('prep', ...), ('reg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True



## 6. Apply the trained model to exoplanets

This produces **predicted mineral group proportions** for each exoplanet with available features and saves a results CSV.


In [25]:

# Prepare exoplanet feature matrix using the same columns
X_exo = df[train_feature_cols].copy()

# Predict (will impute missing values as configured)
Y_pred = model.predict(X_exo)

pred_cols = [f'pred_{t}' for t in ['iron_metal','silicates','water_ice','sulfates_carbonates']]
pred_df = pd.DataFrame(Y_pred, columns=pred_cols)

# Clip to [0,1] and renormalize rows (optional, since regression can output outside range)
pred_df = pred_df.clip(lower=0.0, upper=1.0)
row_sums = pred_df.sum(axis=1).replace(0, np.nan)
pred_df_norm = pred_df.div(row_sums, axis=0).fillna(0.0)

# Save
out_path = Path('exominerals_predictions.csv')
out[out.columns.intersection(list(df.columns) + list(pred_df_norm.columns) + ['ExoMineralIndex'])].to_csv(out_path, index=False)
out_path.as_posix()


'exominerals_predictions.csv'